In [ ]:
import os
import glob
import pandas as pd
from openbabel import pybel
from cclib.io import ccread
from pathlib import Path

In [ ]:
def name_from_path(file):
    return str(file).split('.')[0].split('/')[-1]

In [ ]:
def read_energies(file):
    '''Read the final SCF energy from a Gaussian09 output file'''
    
    data = ccread(file)
    return data.scfenergies[-1]

In [ ]:
def mol_to_angle_length(fname: str):
    '''Read dihedral angles and bond lengths from a Gaussian09 file'''
    
        mol = next(pybel.readfile('g09', fname))
        smarts = pybel.Smarts("[#16R1]@[#6R1]~!@[#6R1]@[#16R1]")
        matches = smarts.findall(mol)
        total_length = 0.0
        total_angle = 0.0
        for counter, match in enumerate(matches):
            if len(match) != 4:
                print("weirdo %d" % len(match))
            # get the atom indexes from the SMARTS match
            a = (match[0])
            b = (match[1])
            c = (match[2])
            d = (match[3])
    
            # print the dihedral angle
            dihedral = mol.OBMol.GetTorsion(a, b, c, d)
            if dihedral < 0:
                dihedral *= -1
            if dihedral < 90:
                dihedral = 180 - dihedral
            length = mol.OBMol.GetBond(b,c).GetLength()
            total_angle += dihedral
            total_length += length
        return total_angle/len(matches), total_length/len(matches)

In [ ]:
def read_round(round_dir):
    '''Read energies and geometries from all files in a single 'round' directory'''
    
    columns = [
        'Molecule', 'Round', 'Neutral Energy', 'Cation Energy', 'Anion Energy',
        'Cation@neutral Energy', 'Anion@neutral Energy', 
        'Neutral@cation Energy', 'Neutral@anion Energy',
        'Neutral Dihedral Angle', 'Neutral Bond Length',
        'Cation Dihedral Angle', 'Cation Bond Length',
        'Anion Dihedral Angle', 'Anion Bond Length'
    ]

    round_info = pd.DataFrame(columns = columns)

    dirs = [f.name for f in os.scandir(round_dir) if (not f.name.startswith('.') and f.is_dir())]

    for dir in dirs:
        files = [os.path.join(round_dir, dir, file) for file in os.listdir(os.path.join(round_dir, dir)) if file.endswith('.g09.gz')]

        for file in files:
            filepath = os.path.join(round_dir, dir, file)
            molecule = name_from_path(file)
            round_id = str(file).split('/')[-3]

            if molecule not in round_info['Molecule'].values:
                round_info = pd.concat([round_info, pd.DataFrame({'Molecule': [molecule], 'Round': [round_id]})], ignore_index = True)
            config = str(file).split('/')[-2].capitalize()
            energy_colname = f"{config} Energy"
            angle_colname = f"{config} Dihedral Angle"
            length_colname = f"{config} Bond Length"

            if len(config) < 10:
                try:
                    angle, length = mol_to_angle_length(filepath)
                    round_info.loc[round_info['Molecule'] == molecule, [angle_colname]] = angle
                    round_info.loc[round_info['Molecule'] == molecule, [length_colname]] = length
                
                except:
                    round_info.loc[round_info['Molecule'] == molecule, [angle_colname]] == ""
                    round_info.loc[round_info['Molecule'] == molecule, [length_colname]] == ""
            
            try:
                energy = read_energies(filepath)
                round_info.loc[round_info['Molecule'] == molecule, [energy_colname]] = energy
            
            except:
                round_info.loc[round_info['Molecule'] == molecule, [energy_colname]] == ""
    
    round_info = round_info.rename(columns = {
        'Neutral@anion Energy': 'Neutral@Anion Energy',
        'Neutral@cation Energy': 'Neutral@Cation Energy',
        'Anion@neutral Energy': 'Anion@Neutral Energy',
        'Cation@neutral Energy': 'Cation@Neutral Energy'
    })
    return round_info

In [ ]:
def calc_energies(df):
    '''Take equilibrium and single-point energies to calculate individual and total λ energies'''
    
    try:
        df['Cation λ0'] = (df['Neutral@Cation Energy'] - df['Neutral Energy']).where(df[['Neutral@Cation Energy', 'Neutral Energy']].notna().all(axis=1))
        df['Cation λ1'] = (df['Cation@Neutral Energy'] - df['Cation Energy']).where(df[['Cation@Neutral Energy', 'Cation Energy']].notna().all(axis=1))
        df['DFT Cation λ'] = (df['Cation λ0'] + df['Cation λ1']).where(df[['Cation λ0', 'Cation λ1']].notna().all(axis=1))
    except:
        pass
    
    try:
        df['Anion λ0'] = (df['Neutral@Anion Energy'] - df['Neutral Energy']).where(df[['Neutral@Anion Energy', 'Neutral Energy']].notna().all(axis=1))
        df['Anion λ1'] = (df['Anion@Neutral Energy'] - df['Anion Energy']).where(df[['Anion@Neutral Energy', 'Anion Energy']].notna().all(axis=1))
        df['DFT Anion λ'] = (df['Anion λ0'] + df['Anion λ1']).where(df[['Anion λ0', 'Anion λ1']].notna().all(axis=1))
    except:
        pass

    return df

In [ ]:
def main_round(round_dir, output_dir, output_name):
    '''Main processing function to read energies from each 'round' directory'''
    
    df = pd.DataFrame(read_round(round_dir))
    df = calc_energies(df)
    df = df.sort_values(by = ['Molecule'], ascending = True, ignore_index = True)
    df.to_csv(os.path.join(output_dir, output_name), index = False)

In [ ]:
def parse_all_rounds(dft_dir, output_dir, output_basename):
    '''Parse energies from multiple 'round' directories'''
    
    round_dirs = [f.name for f in os.scandir(dft_dir) if (not f.name.startswith('.') and f.is_dir() and ('round' in f.name))]
    for round_dir in round_dirs:
        round_path = os.path.join(dft_dir, round_dir)
        round_num = round_dir.split('round')[1]
        output_filename = output_basename + round_num + '.csv'
        main_round(round_path, output_dir, output_filename)
        print(f"Created {output_filename}")

In [ ]:
round_dir = '/ihome/ghutchison/kwl20/reorganization/commonHexamers/RFc/DFT/'
output_dir = os.getcwd()
output_name = 'reorg_cations.csv'
main_round(round_dir, output_dir, output_name)